# Window Detection and Permutation ANOVA

## For Best Frans

In [1]:
from analyses.response_window_analysis import run_permutation_anova_by_window
from analyses.spike_count import extract_spike_counts_from_windows
import pandas as pd
from analyses.response_window_finder.threshold_window_detection import compute_timebinned_spikecount_per_neuron, \
    z_score, threshold_and_fill_gap, extract_consecutive_ranges, remove_consecutive_tuples, \
    find_corresponding_values_for_index_ranges
from tqdm import tqdm
import numpy as np
from analyses.data_readers.recording_metadata_reader import RecordingMetadataReader

In [2]:
# Detect windows for best frans
prelim = RecordingMetadataReader().get_metadata_for_preliminary_analysis()
results = []
bin_size = 0.05  # in sec
rounded_time = np.round(np.arange(bin_size, 3.50, bin_size), 2)
monkey_group = 'Best Frans'
for _, row in tqdm(prelim.iterrows(), total = len(prelim), desc = "Processing each recording day..."):
    round_no = row['Round No.']
    date = str(row['Date'].strftime('%Y-%m-%d'))
    timebin_spikecount_list = compute_timebinned_spikecount_per_neuron(date, round_no, bin_size, monkey_group)
    for _, r in timebin_spikecount_list.iterrows():
        data = r['TotalSpikeCountList']
        neuron = r['NeuronID']
        normalized_data = z_score(data)
        thresh = 0.5
        change_points = threshold_and_fill_gap(normalized_data, thresh)
        windows = extract_consecutive_ranges(change_points)
        filtered_windows = remove_consecutive_tuples(windows)
        time_windows = find_corresponding_values_for_index_ranges(filtered_windows, rounded_time)
        if len(time_windows) > 0:
            for start_time, end_time in time_windows:
                results.append({
                    'NeuronID': neuron,
                    'WindowStart_ms': int(start_time * 1000),
                    'WindowEnd_ms': int(end_time * 1000)
                })


Processing each recording day...: 100%|██████████| 54/54 [02:32<00:00,  2.82s/it]


In [3]:
# Detected windows
bf_results_df = pd.DataFrame(results)
bf_results_df = bf_results_df.sort_values(by=['NeuronID'])
bf_results_df

,NeuronID,WindowStart_ms,WindowEnd_ms
0,2023-09-26_1_Channel.C_005_Unit 1,1700,1850
1,2023-09-26_1_Channel.C_009_Unit 1,450,550
2,2023-09-26_1_Channel.C_009_Unit 1,2000,2150
3,2023-09-26_1_Channel.C_011_Unit 2,1000,1100
4,2023-09-26_1_Channel.C_011_Unit 2,1450,1650
...,...,...,...
813,2023-12-18_3_Channel.C_015_Unit 1,400,500
814,2023-12-18_3_Channel.C_015_Unit 1,850,1050
816,2023-12-18_3_Channel.C_020_Unit 1,200,500
817,2023-12-18_3_Channel.C_026_Unit 1,300,500


In [9]:
# Extract spike counts in the windows
bf_final_df = extract_spike_counts_from_windows(bf_results_df)

Extracting spike counts: 100%|██████████| 819/819 [00:53<00:00, 15.30it/s]


In [17]:
# Run Perm ANOVA on BestFrans
bestfrans_df = bf_final_df[bf_final_df['MonkeyGroup'] == 'Best Frans']
bf_perm_results, bf_sig_results = run_permutation_anova_by_window(bestfrans_df,
                                category_col='MonkeyName',
                                neuron_col='NeuronID',
                                count_col='SpikeCount',
                                window_start_col='WindowStart_ms',
                                window_end_col='WindowEnd_ms',
                                n_permutations=1000,
                                alpha=0.05,
                                plot=False)

Running Perm ANOVA per (Neuron, Window): 100%|██████████| 819/819 [01:15<00:00, 10.91it/s]


All Results:
                              NeuronID  WindowStart_ms  WindowEnd_ms  \
0    2023-09-26_1_Channel.C_005_Unit 1            1700          1850   
1    2023-09-26_1_Channel.C_009_Unit 1             450           550   
2    2023-09-26_1_Channel.C_009_Unit 1            2000          2150   
3    2023-09-26_1_Channel.C_011_Unit 2            1000          1100   
4    2023-09-26_1_Channel.C_011_Unit 2            1450          1650   
..                                 ...             ...           ...   
814  2023-12-18_3_Channel.C_015_Unit 1             850          1050   
815  2023-12-18_3_Channel.C_015_Unit 1            1150          1250   
816  2023-12-18_3_Channel.C_020_Unit 1             200           500   
817  2023-12-18_3_Channel.C_026_Unit 1             300           500   
818  2023-12-18_3_Channel.C_029_Unit 1              50           300   

     F-statistic  p-value  
0       0.592963    0.686  
1       0.605478    0.607  
2       1.621436    0.139  
3       0

In [18]:
bf_sig_results.head()

,NeuronID,WindowStart_ms,WindowEnd_ms,F-statistic,p-value
47,2023-09-26_3_Channel.C_027_Unit 1,250,450,3.282644,0.015
89,2023-09-29_3_Channel.C_029_Unit 1,400,550,3.152212,0.026
119,2023-10-03_3_Channel.C_013_Unit 1,750,850,2.557818,0.039
134,2023-10-03_4_Channel.C_010_Unit 1,100,650,2.706827,0.035
148,2023-10-03_4_Channel.C_025_Unit 2,850,950,3.512962,0.004


In [ ]:
bf_sig_results[['Date', 'Round No.', 'Cell']] = bf_sig_results['NeuronID'].str.split('_', n=2, expand=True)
bf_sig_results['Time Window'] = list(zip(bf_sig_results['WindowStart_ms'], bf_sig_results['WindowEnd_ms']))
bf_sig_results.head() # --- save this and share with Ed !!

In [31]:
# Get spike counts anova passed windows
spike_count_for_sig_windows_anova_passed = extract_spike_counts_from_windows(bf_sig_results)

Extracting spike counts: 100%|██████████| 41/41 [00:28<00:00,  1.45it/s]


In [35]:
spike_count = spike_count_for_sig_windows_anova_passed.copy()
spike_count.head()

,NeuronID,MonkeyName,MonkeyGroup,TaskField,WindowStart_ms,WindowEnd_ms,SpikeCount
0,2023-09-26_3_Channel.C_027_Unit 1,40J,Stranger Things,1695753698845000,250,450,0
1,2023-09-26_3_Channel.C_027_Unit 1,68F,Best Frans,1695753698942000,250,450,3
2,2023-09-26_3_Channel.C_027_Unit 1,68Y,Stranger Things,1695753698996000,250,450,0
3,2023-09-26_3_Channel.C_027_Unit 1,35Y,Instigators,1695753699056000,250,450,3
4,2023-09-26_3_Channel.C_027_Unit 1,DF2I,Stranger Things,1695753699165000,250,450,0


In [41]:
spike_count[['Date', 'Round No.', 'Cell']] = spike_count['NeuronID'].str.split('_', n=2, expand=True)
spike_count['Time Window'] = list(zip(spike_count['WindowStart_ms'], spike_count['WindowEnd_ms']))
spike_count.head()

,NeuronID,MonkeyName,MonkeyGroup,TaskField,WindowStart_ms,WindowEnd_ms,SpikeCount,Date,Round No.,Cell,Time Window
0,2023-09-26_3_Channel.C_027_Unit 1,40J,Stranger Things,1695753698845000,250,450,0,2023-09-26,3,Channel.C_027_Unit 1,"(250, 450)"
1,2023-09-26_3_Channel.C_027_Unit 1,68F,Best Frans,1695753698942000,250,450,3,2023-09-26,3,Channel.C_027_Unit 1,"(250, 450)"
2,2023-09-26_3_Channel.C_027_Unit 1,68Y,Stranger Things,1695753698996000,250,450,0,2023-09-26,3,Channel.C_027_Unit 1,"(250, 450)"
3,2023-09-26_3_Channel.C_027_Unit 1,35Y,Instigators,1695753699056000,250,450,3,2023-09-26,3,Channel.C_027_Unit 1,"(250, 450)"
4,2023-09-26_3_Channel.C_027_Unit 1,DF2I,Stranger Things,1695753699165000,250,450,0,2023-09-26,3,Channel.C_027_Unit 1,"(250, 450)"


In [44]:
group_cols = ['Date', 'Round No.', 'Cell', 'Time Window','MonkeyName']
spike_count['Date'] = spike_count['Date'].astype(str)
grouped_df = spike_count.groupby(group_cols)['SpikeCount'].apply(list).reset_index()
final_spike_count_df = grouped_df.pivot(
    index=['Date', 'Round No.', 'Cell', 'Time Window'],
    columns='MonkeyName',
    values='SpikeCount'
).reset_index()
print(final_spike_count_df.head()) # -- save this and share with Ed!!

### For Zombies

In [ ]:
monkey_group = 'Zombies'
for _, row in tqdm(prelim.iterrows(), total = len(prelim), desc = "Processing each recording day..."):
    round_no = row['Round No.']
    date = str(row['Date'].strftime('%Y-%m-%d'))
    timebin_spikecount_list = compute_timebinned_spikecount_per_neuron(date, round_no, bin_size, monkey_group)
    for _, r in timebin_spikecount_list.iterrows():
        data = r['TotalSpikeCountList']
        neuron = r['NeuronID']
        normalized_data = z_score(data)
        thresh = 0.5
        change_points = threshold_and_fill_gap(normalized_data, thresh)
        windows = extract_consecutive_ranges(change_points)
        filtered_windows = remove_consecutive_tuples(windows)
        time_windows = find_corresponding_values_for_index_ranges(filtered_windows, rounded_time)
        if len(time_windows) > 0:
            print(f"---------------- {neuron} ----------------")
            print(time_windows)
            for start_time, end_time in time_windows:
                results.append({
                    'NeuronID': neuron,
                    'WindowStart_ms': int(start_time * 1000),
                    'WindowEnd_ms': int(end_time * 1000)
                })

zom_results_df = pd.DataFrame(results)
zom_results_df = zom_results_df.sort_values(by=['NeuronID'])
zom_final_df = extract_spike_counts_from_windows(zom_results_df)
zombies_df = zom_final_df[zom_final_df['MonkeyGroup'] == 'Zombies']
zom_perm_results, zom_sig_results = run_permutation_anova_by_window(zombies_df,
                                category_col='MonkeyName',
                                neuron_col='NeuronID',
                                count_col='SpikeCount',
                                window_start_col='WindowStart_ms',
                                window_end_col='WindowEnd_ms',
                                n_permutations=1000,
                                alpha=0.05,
                                plot=False)